# Can Vision Models Read? — Interactive Presentation Demo

This interactive notebook demonstrates the key claims and visual mechanics of the **VISWORD** research project:
1. **The Resolution & Cropping Bottleneck**: Visualizing the sub-pixel legibility loss under downsizing, and how our intelligent `TextAwareCropper` snaps boundaries to whitespace gaps.
2. **Interactive Page Retrieval (Protocol-A)**: Live visual page retrieval comparing **CLIP** (contrastive pretraining), **DINOv2** (self-supervised representation), and **I-JEPA** (predictive pretraining) at native resolution.

## 1. Setup, Repo Clone, and Dependencies

If running in Google Colab, this cell clones the repository, checks out the `dev/v1-legible-reading` branch (which contains the demo dataset and code), sets the working directory, and installs the required packages. ~1-2 minutes.

In [16]:
import os, sys, subprocess

# Detect environment
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    # 1. Clone the repo and setup PYTHONPATH on Colab
    if not os.path.exists('/content/visword'):
        print('Cloning repository on Colab...')
        subprocess.run(['git', 'clone', '--depth', '50',
                        'https://github.com/hkanpak21/Comp447_VISWORD.git', '/content/visword'], check=True)
        # Checkout the active presentation updates branch
        subprocess.run(['git', 'checkout', 'dev/v1-legible-reading'], cwd='/content/visword', check=True)
    os.chdir('/content/visword')
    sys.path.append(os.path.abspath('src'))
    print('Working directory set to Colab path:', os.getcwd())
    print('Installing python dependencies on Colab...')
    subprocess.run(['pip', 'install', '-q', 'open_clip_torch', 'timm', 'transformers',
                    'sentence-transformers', 'ipywidgets'], check=True)
else:
    # Local Valar cluster setup
    # Determine the repository root dynamically (usually 1 level up from 'notebooks' folder or the current dir)
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
    os.chdir(project_root)
    if os.path.exists(os.path.join(project_root, 'src')):
        if os.path.join(project_root, 'src') not in sys.path:
            sys.path.append(os.path.join(project_root, 'src'))
        print('Running locally on Valar cluster. Project root set to:', project_root)
    else:
        print('WARNING: src directory not found at', project_root)
print('Setup complete!')

Running locally on Valar cluster. Project root set to: /scratch/bbakay22/VISWORD
Setup complete!


## 2. Interactive Cropping: Naive vs. Text-Aware Snapping

Standard vision transformers process inputs in fixed $224\times224$ shapes. If we resize a full $980\times980$ page to $224\times224$, the text becomes sub-pixel and completely illegible. 

To solve this resolution bottleneck, we slice pages into crops at **native resolution** ($crop\_size = 490$, target $224$ for standard backbones). 

However, **naive sliding-window cropping** cuts text lines in half, destroying character structure.
Our **`TextAwareCropper`** dynamically calculates the horizontal ink projection of a page and snaps vertical crop boundaries to the centers of whitespace gaps.

Use the dropdown below to select a Wikipedia page, and use the sliders to adjust the detection thresholds. The plot will show:
- **Left**: The full page layout with detected whitespace gaps (shaded gray) and crop boundary cuts (green lines).
- **Right**: A close-up zoom of a boundary comparing the Naive Cut (red, splits letters) vs the Text-Aware Cut (green, snaps to whitespace).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from visword.data.cropper import TextAwareCropper
import json
from pathlib import Path

# Load manifest to get page names and files
manifest_path = 'presentation/demo_data/manifest.json'
manifest = json.loads(Path(manifest_path).read_text())
rows = manifest['rows']
titles = [r['title'] for r in rows]

# Interactive visualizer widgets
page_select = widgets.Dropdown(
    options=[(r['title'], idx) for idx, r in enumerate(rows)],
    value=0,
    description='Page:',
)
whiteness_slider = widgets.IntSlider(
    value=245, min=200, max=255, step=1,
    description='Whiteness Thresh:',
    style={'description_width': 'initial'},
)
ink_slider = widgets.FloatSlider(
    value=0.01, min=0.0, max=0.1, step=0.005,
    description='Line Ink Thresh:',
    style={'description_width': 'initial'},
)
gap_slider = widgets.IntSlider(
    value=2, min=1, max=10, step=1,
    description='Min Gap Height:',
    style={'description_width': 'initial'},
)

visualizer_output = widgets.Output()

def update_visualizer(change=None):
    with visualizer_output:
        clear_output(wait=True)
        idx = page_select.value
        row = rows[idx]
        img_path = 'presentation/demo_data/' + row['image_path']
        im = Image.open(img_path).convert('RGB')
        
        # Instantiate cropper with dynamic parameters
        cropper = TextAwareCropper(
            crop_size=490,
            target_size=224,
            whiteness_threshold=whiteness_slider.value,
            line_ink_threshold=ink_slider.value,
            gap_min_height=gap_slider.value
        )
        
        # Compute row ink and gaps
        row_ink = cropper._row_ink(im)
        gaps = cropper.inter_line_gaps(im)
        y_bands = cropper.y_cuts(im)
        
        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        
        # Left Panel: Full Page with detected gaps and cuts
        axes[0].imshow(im)
        # Draw detected gaps
        for g0, g1 in gaps:
            axes[0].axhspan(g0, g1, color='gray', alpha=0.3)
        # Draw crop boundary cuts (y_bands bottom edges except last)
        for _, bot in y_bands[:-1]:
            axes[0].axhline(y=bot, color='#22c55e', linestyle='--', linewidth=2, label='Text-Aware Cut' if bot == y_bands[0][1] else '')
        
        axes[0].set_title(f'A: Page Layout & Gaps ({im.width}x{im.height})\n(Shaded = Whitespace Gaps, Green lines = Crop Cuts)')
        axes[0].axis('off')
        
        # Right Panel: Zoom on the first cut boundary
        if len(y_bands) > 1:
            cut_y = y_bands[0][1]
            naive_y = cropper.crop_size
            
            # Dynamically adjust zoom box to always contain both cuts
            mid_y = (naive_y + cut_y) // 2
            half_h = max(45, abs(naive_y - cut_y) + 15)
            y0 = max(0, mid_y - half_h)
            y1 = min(im.height, mid_y + half_h)
            
            zoom_box = im.crop((50, y0, 600, y1))
            axes[1].imshow(zoom_box)
            # Draw line coordinates relative to zoom_box
            rel_naive_y = naive_y - y0
            rel_text_y = cut_y - y0
            
            axes[1].axhline(y=rel_naive_y, color='#ef4444', linewidth=2.5, label='Naive Cut (Splits characters)')
            axes[1].axhline(y=rel_text_y, color='#22c55e', linewidth=2.5, label='Text-Aware Cut (Snaps to gap)')
            
            axes[1].set_title(f'B: Zoom-in at Crop Boundary (y={cut_y})')
            axes[1].legend(loc='upper right')
        else:
            axes[1].text(0.5, 0.5, 'Single crop only (No vertical cuts needed)', 
                         horizontalalignment='center', verticalalignment='center', transform=axes[1].transAxes)
            axes[1].set_title('B: Zoom-in Crop Boundary')
            
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

# Observe widget changes
page_select.observe(update_visualizer, names='value')
whiteness_slider.observe(update_visualizer, names='value')
ink_slider.observe(update_visualizer, names='value')
gap_slider.observe(update_visualizer, names='value')

# Trigger initial run
update_visualizer()

# Layout layout
controls = widgets.VBox([
    widgets.HBox([page_select, whiteness_slider]),
    widgets.HBox([ink_slider, gap_slider])
])
display(controls, visualizer_output)

Output()

## 3. Interactive Page Retrieval (Protocol-A)

We now run a live evaluation of the paper's **Protocol-A** page-level retrieval. 
For a query crop $c$ from page $p$:
1. The **Gallery** holds one visual representation per page, defined as the L2-normalized mean of all its crop visual embeddings.
2. To prevent trivial layout-matching cheats on the query's own page, the gallery vector for page $p$ is computed *excluding the query crop $c$ itself* (Leave-One-Crop-Out LOO aggregation).
3. Retrieval computes the cosine similarity between the query crop and the gallery representations to rank the 10 pages.

We load three models:
* **CLIP** (Contrastive vision-language representation)
* **DINOv2** (Self-supervised layout matching)
* **I-JEPA** (Predictive image pretraining)

First, we load the models and compute all crop embeddings for the 10 pages on-the-fly.

In [18]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from visword.analysis.platonic_alignment import encode_dinov2, encode_clip_image, encode_ijepa
from visword.data.cropper import TextAwareCropper
import tempfile
from pathlib import Path
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Load manifest and list page files
manifest_path = 'presentation/demo_data/manifest.json'
manifest = json.loads(Path(manifest_path).read_text())
rows = manifest['rows']

# We crop all 10 pages using TextAwareCropper (size=490, target=224 for models)
cropper = TextAwareCropper(crop_size=490, target_size=224)

# Keep track of crops, page associations, and saved paths
all_crops_pil = []
crop_page_ids = []
crop_img_paths = []

temp_dir = tempfile.TemporaryDirectory()
td_path = Path(temp_dir.name)

for idx, r in enumerate(rows):
    img_path = 'presentation/demo_data/' + r['image_path']
    im = Image.open(img_path).convert('RGB')
    page_crops = cropper.crop(im)
    
    # Store crops
    for c_idx, c in enumerate(page_crops):
        all_crops_pil.append(c)
        crop_page_ids.append(idx)
        
        # Save to temp path for the encoders
        cp = td_path / f'page_{idx}_crop_{c_idx}.png'
        c.save(cp, format='PNG')
        crop_img_paths.append(cp)

crop_page_ids = np.array(crop_page_ids)

print(f'Generated {len(all_crops_pil)} crops from {len(rows)} pages.')
print('Computing embeddings on-the-fly...')

# Run image encoders
print('  Encoding DINOv2...', flush=True)
feats_dinov2 = torch.from_numpy(encode_dinov2(crop_img_paths, device))
print('  Encoding CLIP...', flush=True)
feats_clip = torch.from_numpy(encode_clip_image(crop_img_paths, device))
print('  Encoding I-JEPA...', flush=True)
feats_ijepa = torch.from_numpy(encode_ijepa(crop_img_paths, device))

print('Embeddings computed:')
print('  DINOv2 shape:', feats_dinov2.shape)
print('  CLIP shape:', feats_clip.shape)
print('  I-JEPA shape:', feats_ijepa.shape)

Using device: cuda
Generated 39 crops from 10 pages.
Computing embeddings on-the-fly...
  Encoding DINOv2...


Using cache found in /home/bbakay22/.cache/torch/hub/facebookresearch_dinov2_main


  Encoding CLIP...
  Encoding I-JEPA...
Embeddings computed:
  DINOv2 shape: torch.Size([39, 768])
  CLIP shape: torch.Size([39, 512])
  I-JEPA shape: torch.Size([39, 1280])


### Live Protocol-A Retrieval Interface

Choose a page and click on one of its unmasked crop thumbnails to select it as the query crop.
When you click **Run Retrieval Query**, the models will compare the query crop against the page gallery using the leave-one-crop-out similarity metric.

In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import ImageDraw

# Create page dropdown
retrieval_page_select = widgets.Dropdown(
    options=[(r['title'], idx) for idx, r in enumerate(rows)],
    value=0,
    description='Target Page:',
    style={'description_width': 'initial'},
)

# Output area for crop thumbnails and results
crop_selection_output = widgets.Output()
retrieval_results_output = widgets.Output()

selected_crop_idx = [0]  # Store selection in a list so it's mutable

def display_crop_options(change=None):
    with crop_selection_output:
        clear_output(wait=True)
        page_idx = retrieval_page_select.value
        
        # Get crop indices for this page
        idxs = np.where(crop_page_ids == page_idx)[0]
        
        print('Select a Query Crop:')
        # Render crops side-by-side with buttons
        crop_boxes = []
        for local_idx, idx in enumerate(idxs):
            img = all_crops_pil[idx]
            
            # Resize image to show as small thumbnail
            thumb = img.resize((100, 100))
            
            # Save thumbnail to a temp file and display it
            out_img = widgets.Image(value=thumb._repr_png_(), width=100, height=100)
            
            btn = widgets.Button(
                description=f'Select Crop {local_idx + 1}',
                button_style='info' if idx == selected_crop_idx[0] else '',
                layout=widgets.Layout(width='100px')
            )
            
            # Capture variable in closure
            def make_handler(global_idx):
                def on_click(b):
                    selected_crop_idx[0] = global_idx
                    display_crop_options() # Redraw to update button styles
                return on_click
            
            btn.on_click(make_handler(idx))
            
            crop_boxes.append(widgets.VBox([out_img, btn]))
        
        display(widgets.HBox(crop_boxes))

def compute_loo_similarity(query_idx, feats):
    """Compute Protocol-A cosine similarities with leave-one-out aggregation."""
    # query_idx: index in all_crops_pil
    # feats: (N, D) torch.Tensor (L2-normed)
    N, D = feats.shape
    q_pid = crop_page_ids[query_idx]
    
    # Calculate average embeddings for all 10 pages
    sums = torch.zeros(10, D, dtype=feats.dtype)
    counts = torch.zeros(10, dtype=torch.long)
    
    for idx in range(N):
        pid = crop_page_ids[idx]
        sums[pid] += feats[idx]
        counts[pid] += 1
        
    # Standard page average for all pages
    gallery = F.normalize(sums, dim=1, eps=1e-12)
    
    # Recompute LOO average for query's own page
    if counts[q_pid] >= 2:
        loo_sum = sums[q_pid] - feats[query_idx]
        gallery[q_pid] = F.normalize(loo_sum, dim=0, eps=1e-12)
    else:
        # Fallback if page only has 1 crop
        gallery[q_pid] = feats[query_idx]
        
    # Cosine similarities
    sims = (feats[query_idx] @ gallery.T).cpu().numpy()
    return sims

def run_retrieval_query(b):
    with retrieval_results_output:
        clear_output(wait=True)
        q_idx = selected_crop_idx[0]
        target_page_idx = crop_page_ids[q_idx]
        
        # Run similarity calculations
        sims_clip = compute_loo_similarity(q_idx, feats_clip)
        sims_ijepa = compute_loo_similarity(q_idx, feats_ijepa)
        sims_dinov2 = compute_loo_similarity(q_idx, feats_dinov2)
        
        # Rank the pages
        rank_clip = np.argsort(-sims_clip)
        rank_ijepa = np.argsort(-sims_ijepa)
        rank_dinov2 = np.argsort(-sims_dinov2)
        
        # Plot query crop
        fig, axes = plt.subplots(1, 4, figsize=(18, 5.5))
        
        # Plot query crop image
        query_img = all_crops_pil[q_idx]
        axes[0].imshow(query_img)
        c_num = np.where(crop_page_ids == target_page_idx)[0].tolist().index(q_idx)+1
        axes[0].set_title(f'Query Crop (Crop {c_num})', fontsize=10)
        axes[0].axis('off')
        
        # Plot CLIP Top retrieve
        best_clip_idx = rank_clip[0]
        best_clip_row = rows[best_clip_idx]
        clip_correct = 'CORRECT ✓' if best_clip_idx == target_page_idx else 'WRONG ✗'
        clip_img = Image.open('presentation/demo_data/' + best_clip_row['image_path']).convert('RGB')
        axes[1].imshow(clip_img)
        axes[1].set_title(f'CLIP: {sims_clip[best_clip_idx]:.3f} ({clip_correct})', fontsize=10)
        axes[1].axis('off')
        
        # Plot DINOv2 Top retrieve
        best_dinov2_idx = rank_dinov2[0]
        best_dinov2_row = rows[best_dinov2_idx]
        dinov2_correct = 'CORRECT ✓' if best_dinov2_idx == target_page_idx else 'WRONG ✗'
        dinov2_img = Image.open('presentation/demo_data/' + best_dinov2_row['image_path']).convert('RGB')
        axes[2].imshow(dinov2_img)
        axes[2].set_title(f'DINOv2: {sims_dinov2[best_dinov2_idx]:.3f} ({dinov2_correct})', fontsize=10)
        axes[2].axis('off')
        
        # Plot I-JEPA Top retrieve
        best_ijepa_idx = rank_ijepa[0]
        best_ijepa_row = rows[best_ijepa_idx]
        ijepa_correct = 'CORRECT ✓' if best_ijepa_idx == target_page_idx else 'WRONG ✗'
        ijepa_img = Image.open('presentation/demo_data/' + best_ijepa_row['image_path']).convert('RGB')
        axes[3].imshow(ijepa_img)
        axes[3].set_title(f'I-JEPA: {sims_ijepa[best_ijepa_idx]:.3f} ({ijepa_correct})', fontsize=10)
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Leaderboard tables
        col_width = 38
        def format_rank_entry(rank, idx, score):
            title = rows[idx]['title']
            if len(title) > 20:
                title = title[:17] + '...'
            entry_str = f'Rank {rank}: "{title}"'
            return f'{entry_str:<28} ({score:.3f})'
            
        print(f"{'--- CLIP Ranks ---':<{col_width}}{'--- DINOv2 Ranks ---':<{col_width}}{'--- I-JEPA Ranks ---':<{col_width}}")
        for r in range(3):
            clip_line = format_rank_entry(r+1, rank_clip[r], sims_clip[rank_clip[r]])
            dinov2_line = format_rank_entry(r+1, rank_dinov2[r], sims_dinov2[rank_dinov2[r]])
            ijepa_line = format_rank_entry(r+1, rank_ijepa[r], sims_ijepa[rank_ijepa[r]])
            print(f"{clip_line:<{col_width}}{dinov2_line:<{col_width}}{ijepa_line:<{col_width}}")

# Wire events
retrieval_page_select.observe(display_crop_options, names='value')

btn_run = widgets.Button(
    description='Run Retrieval Query',
    button_style='primary',
    layout=widgets.Layout(width='200px', margin='10px 0px 10px 0px')
)
btn_run.on_click(run_retrieval_query)

# Initial draw
display_crop_options()

# Display widgets
display(
    retrieval_page_select,
    crop_selection_output,
    btn_run,
    retrieval_results_output
)

Dropdown(description='Target Page:', options=(('Hercule Poirot', 0), ('Eiffel', 1), ('Glucono delta-lactone', …

Output()

Button(button_style='primary', description='Run Retrieval Query', layout=Layout(margin='10px 0px 10px 0px', wi…

Output()